# ZK-KGVerify v2 - Colab T4 reproduction (with disconnect-safe checkpoints)

Reproduces every number in the paper using the upgraded code:

- **Real BN128 elliptic-curve crypto** (py_ecc) for Pedersen + Schnorr-Fiat-Shamir
- Deterministic seeding (`RANDOM_SEED=42`)
- Full filtered evaluation over the 20,466 FB15k-237 test triples
- 1,000 ZK proofs + verifications + tamper test
- Local Python blockchain simulation (real Sepolia numbers come from `run_sepolia.py`)
- **Drive-backed checkpoints** so a Colab disconnect mid-run does NOT waste prior training

**Hardware**: Runtime > Change runtime type > **T4 GPU**.

**If the runtime disconnects:** just open this notebook again, `Runtime > Run all`. The checkpoint logic skips any model that already trained and evaluated in a previous session.

In [ ]:
# 1. Mount Google Drive FIRST -- so checkpoints survive disconnects.
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/ZK-KGVerify-checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/ZK-KGVerify-results-v2'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_DIR)
print('Final results will be saved to:', RESULTS_DIR)
print('Existing checkpoints:', os.listdir(CKPT_DIR) if os.path.isdir(CKPT_DIR) else '(none)')

In [ ]:
# 2. Clone the upgraded repo (v2 branch)
if not os.path.isdir('ZK-KGVerify'):
    !git clone -b v2-real-bn128 https://github.com/Sanjoy-Chattopadhay/ZK-KGVerify.git
%cd ZK-KGVerify

In [ ]:
# 3. Install deps.
!pip install -q py_ecc==8.0.0 tqdm
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 4. Point the pipeline at Drive-backed checkpoint and results directories.
#    This OVERRIDES the defaults in configs/config.py for this Colab session.
import configs.config as cfg
cfg.CHECKPOINT_DIR = CKPT_DIR
cfg.RESULTS_DIR = RESULTS_DIR
cfg.RESUME = True
print('CHECKPOINT_DIR =', cfg.CHECKPOINT_DIR)
print('RESULTS_DIR    =', cfg.RESULTS_DIR)
print('RESUME         =', cfg.RESUME)

In [ ]:
# 5. Sanity-check the BN128 ZKP layer before any heavy compute.
import sys; sys.path.insert(0, '.')
import numpy as np
from src.zkp_module import batch_generate_proofs, batch_verify_proofs
np.random.seed(42)
vecs = [np.random.randn(384).astype('float32') for _ in range(50)]
scores = [float(i*0.1) for i in range(50)]
triples = [(i,(i*7)%237,(i*13)%14541) for i in range(50)]
ps, _ = batch_generate_proofs(vecs, scores, triples, 'sanity')
rs, vs = batch_verify_proofs(ps)
print(f'BN128 ZKP sanity: {vs["num_valid"]}/{vs["num_verified"]} valid, avg verify {vs["avg_verify_time"]*1000:.1f}ms')
assert vs['num_valid'] == vs['num_verified'], 'ZKP broken -- abort'

In [ ]:
# 6. Run the full pipeline. Skipping is automatic for models that already
#    have a checkpoint in CKPT_DIR. Expected wall-clock on T4 (cold start):
#    ~50-90 min. On warm restart after disconnect: skip straight to ZKP/BC.
from src.pipeline import run_full_pipeline
results = run_full_pipeline()

In [ ]:
# 7. Print the numbers we actually need for the paper.
import json
all_results_path = os.path.join(cfg.RESULTS_DIR, 'all_results.json')
with open(all_results_path) as f:
    R = json.load(f)
print('--- Link prediction (filtered, full 20,466 test triples) ---')
for m, d in R['link_prediction_metrics'].items():
    print(f"  {m:<8} MRR={d['MRR']:.4f}  H@1={d['Hits@1']:.4f}  H@3={d['Hits@3']:.4f}  H@10={d['Hits@10']:.4f}  (eval n={d['num_evaluated']})")
Z = R['zkp_statistics']
print()
print('--- BN128 ZKP overhead (1000 proofs) ---')
print(f"  Proof gen   avg = {Z['avg_gen_time']*1000:.2f} ms (sd {Z['std_gen_time']*1000:.2f})")
print(f"  Verify      avg = {Z['avg_verify_time']*1000:.2f} ms (sd {Z['std_verify_time']*1000:.2f})")
print(f"  Proof size  avg = {Z['avg_proof_size_bytes']:.0f} bytes")
print(f"  Valid       = {Z['num_valid']}/{Z['num_verified']} ({Z['verification_rate']*100:.1f}%)")
print(f"  Tamper det. = {Z.get('tamper_detection_rate', 0)*100:.1f}%")
B = R['blockchain_statistics']
print()
print('--- Python blockchain simulation ---')
print(f"  Blocks: {B['num_blocks']}  Tx: {B['total_transactions']}  Avg gas/tx: {B['avg_gas_per_tx']:,.0f}")
print(f"  Chain valid: {B['chain_valid']}")